# Phase 2.4: Leakage Detection
**DNA Gene Mapping Project - ML Phase**  
**Author:** Sharique Mohammad  
**Date:** February 2026

## Objective
Identify and remove features that leak target information

## Critical Importance
Leakage = features that contain information about the target that wouldn't be available at prediction time
- Causes artificially high accuracy in training
- Model fails completely in production
- MUST be detected and removed

## Key Tasks
1. Identify obvious target leakage
2. Check for derived features
3. Detect temporal leakage
4. Validate train/test split integrity
5. Create clean feature lists

## Deliverables
- List of leakage features to remove
- Clean feature sets for modeling
- Leakage detection report

## Setup

In [ ]:
# Imports
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from pathlib import Path
import os
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path().absolute().parent.parent
REPORTS_DIR = PROJECT_ROOT / 'data' / 'analytical' / 'reports'

np.random.seed(42)

print("Setup complete")
print(f"Reports: {REPORTS_DIR}")

In [ ]:
# Database connection
load_dotenv()

POSTGRES_HOST = os.getenv('POSTGRES_HOST', 'localhost')
POSTGRES_PORT = os.getenv('POSTGRES_PORT', '5432')
POSTGRES_DB = os.getenv('POSTGRES_DB', 'genome_db')
POSTGRES_USER = os.getenv('POSTGRES_USER', 'postgres')
POSTGRES_PASSWORD = os.getenv('POSTGRES_PASSWORD')

conn_str = f"postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
engine = create_engine(conn_str)

print("Database connection established")

## 1. Load Table Schemas

In [ ]:
# Load column names for each table
print("Loading table schemas...\n")

tables = [
    'clinical_ml_features',
    'disease_ml_features',
    'pharmacogene_ml_features',
    'variant_impact_ml_features',
    'structural_variant_ml_features'
]

table_columns = {}

for table_name in tables:
    query = f"SELECT * FROM gold.{table_name} LIMIT 0"
    df = pd.read_sql(query, engine)
    table_columns[table_name] = df.columns.tolist()
    print(f"{table_name}: {len(df.columns)} columns")

print(f"\nLoaded schemas for {len(table_columns)} tables")

## 2. Identify Obvious Target Leakage

In [ ]:
# Define leakage patterns for each ML task

# Task 1: Variant Pathogenicity Prediction
# Target: target_is_pathogenic vs target_is_benign
# Leakage: Any feature derived from clinical significance

variant_leakage_patterns = [
    'target_is_pathogenic',
    'target_is_benign',
    'target_is_vus',
    'is_pathogenic',
    'is_benign',
    'is_vus',
    'clinical_significance_simple',
    'clinvar_pathogenicity_class',
    'clinical_sig_is_uncertain',
    'predicted_sv_pathogenicity'
]

# Task 2: Structural Variant Risk Prediction
# Target: is_high_risk_sv
# Leakage: Any feature derived from risk assessment

sv_leakage_patterns = [
    'is_high_risk_sv',
    'predicted_sv_pathogenicity'
]

print("Identifying obvious target leakage...\n")
print("Task 1: Variant Pathogenicity Prediction")
print("-"*80)
print("Target columns: target_is_pathogenic, target_is_benign")
print("\nLeakage features to remove:")
for pattern in variant_leakage_patterns:
    print(f"  - {pattern}")

print("\n" + "="*80)
print("Task 2: Structural Variant Risk Prediction")
print("-"*80)
print("Target column: is_high_risk_sv")
print("\nLeakage features to remove:")
for pattern in sv_leakage_patterns:
    print(f"  - {pattern}")

## 3. Detect Leakage Features in Each Table

In [ ]:
# Scan each table for leakage features
print("\nScanning tables for leakage features...\n")

leakage_features = {}

for table_name, columns in table_columns.items():
    leaked = []
    
    # Check for variant pathogenicity leakage
    for pattern in variant_leakage_patterns:
        if pattern in columns:
            leaked.append((pattern, 'Variant pathogenicity target/derived'))
    
    # Check for SV risk leakage
    for pattern in sv_leakage_patterns:
        if pattern in columns and pattern not in [item[0] for item in leaked]:
            leaked.append((pattern, 'SV risk target/derived'))
    
    if leaked:
        leakage_features[table_name] = leaked
        print(f"{table_name}:")
        print(f"  Leakage features: {len(leaked)}")
        for feat, reason in leaked:
            print(f"    {feat:<45} ({reason})")
        print()
    else:
        print(f"{table_name}: No leakage features detected\n")

total_leakage = sum(len(v) for v in leakage_features.values())
print(f"Total leakage features across all tables: {total_leakage}")

## 4. Check for Suspicious Correlations with Target

In [ ]:
# Load sample data and check correlation with targets
# Features with r > 0.99 with target are likely leakage

print("Checking for suspicious correlations with targets...\n")

suspicious_features = {}
SUSPICIOUS_CORRELATION_THRESHOLD = 0.99

# Check clinical_ml_features (variant pathogenicity)
query = "SELECT * FROM gold.clinical_ml_features TABLESAMPLE SYSTEM (2)"
df = pd.read_sql(query, engine)

if 'target_is_pathogenic' in df.columns and 'target_is_benign' in df.columns:
    # Create binary target
    df_binary = df[(df['target_is_pathogenic'] == True) | (df['target_is_benign'] == True)].copy()
    df_binary['target'] = df_binary['target_is_pathogenic'].astype(int)
    
    numeric_cols = df_binary.select_dtypes(include=[np.number]).columns.tolist()
    numeric_cols = [col for col in numeric_cols if col != 'target' and 'id' not in col.lower()]
    
    if len(df_binary) > 100 and len(numeric_cols) > 0:
        correlations = df_binary[numeric_cols + ['target']].corr()['target'].abs().sort_values(ascending=False)
        suspicious = correlations[correlations > SUSPICIOUS_CORRELATION_THRESHOLD].drop('target', errors='ignore')
        
        if len(suspicious) > 0:
            suspicious_features['clinical_ml_features'] = suspicious.to_dict()
            print(f"clinical_ml_features:")
            print(f"  Suspicious correlations with target (|r| > {SUSPICIOUS_CORRELATION_THRESHOLD}): {len(suspicious)}")
            for feat, corr in suspicious.items():
                print(f"    {feat:<45} (r={corr:.4f})")
            print()
        else:
            print(f"clinical_ml_features: No suspicious correlations\n")
    else:
        print("clinical_ml_features: Insufficient data for correlation check\n")

# Check structural_variant_ml_features (SV risk)
query_sv = "SELECT * FROM gold.structural_variant_ml_features TABLESAMPLE SYSTEM (5)"
df_sv = pd.read_sql(query_sv, engine)

if 'is_high_risk_sv' in df_sv.columns:
    df_sv['target'] = df_sv['is_high_risk_sv'].astype(int)
    
    numeric_cols = df_sv.select_dtypes(include=[np.number]).columns.tolist()
    numeric_cols = [col for col in numeric_cols if col != 'target' and 'id' not in col.lower()]
    
    if len(df_sv) > 100 and len(numeric_cols) > 0:
        correlations = df_sv[numeric_cols + ['target']].corr()['target'].abs().sort_values(ascending=False)
        suspicious_sv = correlations[correlations > SUSPICIOUS_CORRELATION_THRESHOLD].drop('target', errors='ignore')
        
        if len(suspicious_sv) > 0:
            suspicious_features['structural_variant_ml_features'] = suspicious_sv.to_dict()
            print(f"structural_variant_ml_features:")
            print(f"  Suspicious correlations with target (|r| > {SUSPICIOUS_CORRELATION_THRESHOLD}): {len(suspicious_sv)}")
            for feat, corr in suspicious_sv.items():
                print(f"    {feat:<45} (r={corr:.4f})")
            print()
        else:
            print(f"structural_variant_ml_features: No suspicious correlations\n")
    else:
        print("structural_variant_ml_features: Insufficient data for correlation check\n")

total_suspicious = sum(len(v) for v in suspicious_features.values())
print(f"Total suspicious features: {total_suspicious}")

## 5. Identify ID and Metadata Columns

In [ ]:
# Identify columns that should not be used as features
# IDs, positions, names, etc.

print("Identifying ID and metadata columns...\n")

id_metadata_patterns = [
    'id', 'name', 'symbol', 'description', 'study_id',
    'position', 'start_pos', 'end_pos', 'chromosome',
    'assembly', 'variant_name', 'sv_id', 'variant_id'
]

id_metadata_features = {}

for table_name, columns in table_columns.items():
    metadata = []
    
    for col in columns:
        col_lower = col.lower()
        for pattern in id_metadata_patterns:
            if pattern in col_lower:
                metadata.append(col)
                break
    
    if metadata:
        id_metadata_features[table_name] = metadata
        print(f"{table_name}:")
        print(f"  ID/metadata columns: {len(metadata)}")
        for col in metadata:
            print(f"    - {col}")
        print()

total_metadata = sum(len(v) for v in id_metadata_features.values())
print(f"Total ID/metadata columns: {total_metadata}")
print("\nNote: These should be excluded from features but kept for indexing")

## 6. Summary and Final Feature Lists

In [ ]:
# Compile all features to remove
print("Compiling final exclusion lists...\n")

all_exclusions = {}

for table_name in table_columns.keys():
    exclusions = set()
    
    # Add leakage features
    if table_name in leakage_features:
        exclusions.update([feat for feat, reason in leakage_features[table_name]])
    
    # Add suspicious features
    if table_name in suspicious_features:
        exclusions.update(suspicious_features[table_name].keys())
    
    # Add ID/metadata
    if table_name in id_metadata_features:
        exclusions.update(id_metadata_features[table_name])
    
    if exclusions:
        all_exclusions[table_name] = sorted(list(exclusions))

# Create summary
summary_data = []

for table_name in table_columns.keys():
    total_cols = len(table_columns[table_name])
    leakage = len([f for f, r in leakage_features.get(table_name, [])])
    suspicious = len(suspicious_features.get(table_name, {}))
    metadata = len(id_metadata_features.get(table_name, []))
    total_exclude = len(all_exclusions.get(table_name, []))
    clean_features = total_cols - total_exclude
    
    summary_data.append({
        'Table': table_name.replace('_ml_features', ''),
        'Total Columns': total_cols,
        'Leakage': leakage,
        'Suspicious': suspicious,
        'ID/Metadata': metadata,
        'Total Exclude': total_exclude,
        'Clean Features': clean_features
    })

summary_df = pd.DataFrame(summary_data)

print("Leakage Detection Summary:")
print("="*100)
print(summary_df.to_string(index=False))
print("="*100)

print(f"\nOverall Statistics:")
print(f"  Total columns: {summary_df['Total Columns'].sum()}")
print(f"  Leakage features: {summary_df['Leakage'].sum()}")
print(f"  Suspicious features: {summary_df['Suspicious'].sum()}")
print(f"  ID/metadata columns: {summary_df['ID/Metadata'].sum()}")
print(f"  Total to exclude: {summary_df['Total Exclude'].sum()}")
print(f"  Clean features for modeling: {summary_df['Clean Features'].sum()}")

## 7. Generate Leakage Detection Report

In [ ]:
# Generate comprehensive leakage detection report
report_path = REPORTS_DIR / 'leakage_detection_report.txt'

with open(report_path, 'w') as f:
    f.write("="*80 + "\n")
    f.write("LEAKAGE DETECTION REPORT\n")
    f.write("DNA Gene Mapping Project - Phase 2.4\n")
    f.write("="*80 + "\n\n")
    
    f.write("SUMMARY\n")
    f.write("-"*80 + "\n")
    f.write(summary_df.to_string(index=False))
    f.write("\n\n")
    
    f.write("OVERALL STATISTICS\n")
    f.write("-"*80 + "\n")
    f.write(f"Total columns: {summary_df['Total Columns'].sum()}\n")
    f.write(f"Leakage features: {summary_df['Leakage'].sum()}\n")
    f.write(f"Suspicious features: {summary_df['Suspicious'].sum()}\n")
    f.write(f"ID/metadata: {summary_df['ID/Metadata'].sum()}\n")
    f.write(f"Total to exclude: {summary_df['Total Exclude'].sum()}\n")
    f.write(f"Clean features: {summary_df['Clean Features'].sum()}\n\n")
    
    f.write("DETAILED FINDINGS\n")
    f.write("="*80 + "\n\n")
    
    if leakage_features:
        f.write("1. TARGET LEAKAGE FEATURES\n")
        f.write("-"*80 + "\n")
        for table, features in leakage_features.items():
            f.write(f"\n{table}:\n")
            for feat, reason in features:
                f.write(f"  - {feat:<45} ({reason})\n")
        f.write("\n")
    
    if suspicious_features:
        f.write("2. SUSPICIOUS CORRELATIONS WITH TARGET\n")
        f.write("-"*80 + "\n")
        for table, features in suspicious_features.items():
            f.write(f"\n{table}:\n")
            for feat, corr in features.items():
                f.write(f"  - {feat:<45} (r={corr:.4f})\n")
        f.write("\n")
    
    if id_metadata_features:
        f.write("3. ID AND METADATA COLUMNS\n")
        f.write("-"*80 + "\n")
        for table, features in id_metadata_features.items():
            f.write(f"\n{table}:\n")
            for feat in features:
                f.write(f"  - {feat}\n")
        f.write("\n")
    
    f.write("COMPLETE EXCLUSION LISTS\n")
    f.write("="*80 + "\n")
    for table, exclusions in all_exclusions.items():
        f.write(f"\n{table}:\n")
        for feat in exclusions:
            f.write(f"  - {feat}\n")
    
    f.write("\n" + "="*80 + "\n")
    f.write("CRITICAL WARNINGS\n")
    f.write("="*80 + "\n")
    f.write("1. NEVER use target columns as features\n")
    f.write("2. NEVER use clinical_significance_simple or derived columns\n")
    f.write("3. ID columns should be used for indexing only, not features\n")
    f.write("4. Verify train/test splits do not leak information\n")
    f.write("\n")
    f.write("NEXT STEPS\n")
    f.write("-"*80 + "\n")
    f.write("- Proceed to Phase 2.5: Feature Reduction\n")
    f.write("- Apply all filters from Phases 2.1-2.4\n")
    f.write("- Create final clean feature lists for modeling\n")

# Save exclusion lists as CSV
for table_name, exclusions in all_exclusions.items():
    exclusion_df = pd.DataFrame({'excluded_feature': exclusions})
    filename = f"{table_name}_exclusions.csv"
    exclusion_df.to_csv(REPORTS_DIR / filename, index=False)
    print(f"Saved: {REPORTS_DIR / filename}")

print(f"\nReport saved: {report_path}")
print("\n" + "="*80)
print("PHASE 2.4 COMPLETE - Leakage Detection")
print("="*80)
print("\nLeakage Findings:")
print(f"  Target leakage: {summary_df['Leakage'].sum()} features")
print(f"  Suspicious correlations: {summary_df['Suspicious'].sum()} features")
print(f"  ID/metadata: {summary_df['ID/Metadata'].sum()} columns")
print(f"\nTotal to exclude: {summary_df['Total Exclude'].sum()}")
print(f"Clean features remaining: {summary_df['Clean Features'].sum()}")
print("\nNext: Phase 2.5 - Feature Reduction")